### Imports

In [1]:
import os
import pandas as pd
import numpy as np
from relaiss import constants
import relaiss as rl
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from alerce.core import Alerce

al = Alerce() 

In [2]:
!pip install alerce 
from alerce.core import Alerce
al = Alerce()  

### ANTARES Query + LAISS Pre-Filter

The cells below call the ANTARES API and run the three-stage LAISS pre-filter (`laiss_prefilter.py`) before passing candidates into the Isolation Forest pipeline:

1. **Stage 1 – Query-level filter** (Elasticsearch): MJD window (last 1 000 days),    `num_mag_values` in [4, 2000].
2. **Stage 2 – Catalog rejection**: excludes loci cross-matched to SDSS stars,    Gaia EDR3, VSX, LINEAR, Véron AGN/QSO, Milliquas, bright guide stars    (class==0), and Gaia DR3 parallax detections.
3. **Stage 3 – Light-curve quality cuts**: multi-band, ≥3 unique epochs,    astrometric stability, and per-band variability thresholds.

In [ ]:
# Install / upgrade the ANTARES client if needed
!pip install antares-client --quiet

In [5]:
import antares_client
from antares_client import search as antares_search
from antares_client.models import Locus
from astropy.coordinates import SkyCoord
from astropy.coordinates import Angle
import astropy.units as u
import pandas as pd
import numpy as np
from astropy.time import Time

In [7]:
# ── ANTARES Elasticsearch query (Stage 1: LAISS query-level pre-filter) ──
#
# Mirrors QUERY_PREFILTER from laiss_prefilter.py:
#   • Only loci with their most-recent alert within the last 1000 days
#   • num_mag_values between 4 and 2000
#
# Additionally the must_not block implements Stage 2 (catalog rejection),
# excluding loci cross-matched to known-star / AGN / QSO catalogs.

MJD_NOW = Time.now().mjd
MJD_CUTOFF = MJD_NOW - 1000        # alerts within the last 1000 days

EXCLUDED_CATALOGS = [
    "sdss_stars",
    "gaia_edr3_distances_bailer_jones",
    "vsx",
    "linear_ll",
    "veron_agn_qso",
    "milliquas",
]

antares_query = {
    "query": {
        "bool": {
            "filter": [
                # most-recent alert within the last 1000 days
                {"range": {"properties.newest_alert_observation_time": {"gte": MJD_CUTOFF}}},
                # at least 4 magnitude measurements
                {"range": {"properties.num_mag_values": {"gte": 4, "lte": 2000}}},
            ],
            "must_not": [
                # reject loci cross-matched to known-star / AGN catalogs
                *[{"term": {"catalogs": cat}} for cat in EXCLUDED_CATALOGS],
            ]
        }
    }
}

print("ANTARES ES query built.")
print(f"  MJD cutoff : {MJD_CUTOFF:.2f}  (now={MJD_NOW:.2f})")

ANTARES ES query built.
  MJD cutoff : 60159.85  (now=61159.85)


In [9]:
# ── Fetch loci from ANTARES and collect light curves ─────────────────────
#
# antares_search.search() returns an iterator; we stop after MAX_LOCI to
# keep the run tractable.  Remove or raise MAX_LOCI for production runs.

MAX_LOCI = 500   # adjust as needed

raw_loci = []
for locus in antares_search.search(antares_query):
    raw_loci.append(locus)
    if len(raw_loci) >= MAX_LOCI:
        break

print(f"Retrieved {len(raw_loci)} loci from ANTARES before light-curve filtering.")

Retrieved 500 loci from ANTARES before light-curve filtering.


In [15]:
# ── Stage 2 (post-retrieval) + Stage 3: LAISS light-curve quality cuts ───
#
# Applies the remaining catalog rejections that can't be expressed in ES
# (Gaia DR3 parallax, bright guide star classification) and the full
# standard_quality_check() from laiss_prefilter.py.

def passes_catalog_rejection(locus: Locus) -> bool:
    """Stage 2 post-retrieval catalog checks."""
    # Reject if Gaia DR3 parallax is present (likely a star)
    gaia_objs = locus.catalog_objects.get("gaia_dr3_source", [])
    for obj in gdr3 if (gdr3 := gaia_objs) else []:
        plx = getattr(obj, "parallax", None)
        if plx is not None and not (isinstance(plx, float) and np.isnan(plx)):
            return False
    # Reject if bright guide-star catalog match has classification == 0 (star)
    bsc_objs = locus.catalog_objects.get("bright_guide_star_catalog", [])
    for obj in bsc_objs:
        if getattr(obj, "classification", None) == 0:
            return False
    return True


def standard_quality_check(ts: pd.DataFrame) -> bool:
    """Stage 3 light-curve quality cuts (from laiss_prefilter.py)."""
    if len(ts['ant_passband'].unique()) < 2:
        return False
    if len(ts['ant_mjd'].round().unique()) < 3:
        return False
    if ts['ant_ra'].std() > 0.5 / 3600.:
        return False
    if ts['ant_dec'].std() > 0.5 / 3600.:
        return False
    for b in ts['ant_passband'].unique():
        sub = ts[ts['ant_passband'] == b]
        if (len(sub) > 1) and (np.ptp(sub['ant_mag']) < 0.2):
            return False
        if len(sub) < 5:
            continue
        if np.ptp(sub['ant_mag']) < 3 * sub['ant_magerr'].mean():
            return False
        if sub['ant_mag'].std() < sub['ant_magerr'].mean():
            return False
        if np.ptp(sub['ant_mag']) < 0.5:
            return False
    return True


passed_loci = []
rejected_counts = {"catalog": 0, "lc_quality": 0, "no_timeseries": 0}

for locus in raw_loci:
    # Stage 2 post-retrieval
    if not passes_catalog_rejection(locus):
        rejected_counts["catalog"] += 1
        continue
    # Stage 3 light-curve quality
    ts = locus.timeseries.to_pandas()
    if ts is None:
        rejected_counts["no_timeseries"] += 1
        continue
    required_cols = {'ant_passband', 'ant_mjd', 'ant_mag', 'ant_magerr', 'ant_ra', 'ant_dec'}
    if not required_cols.issubset(ts.columns):
        rejected_counts["no_timeseries"] += 1
        continue
    if not standard_quality_check(ts):
        rejected_counts["lc_quality"] += 1
        continue
    passed_loci.append(locus)

print(f"Loci passing all LAISS pre-filters : {len(passed_loci)}")
print(f"  Rejected (catalog)               : {rejected_counts['catalog']}")
print(f"  Rejected (LC quality)            : {rejected_counts['lc_quality']}")
print(f"  Rejected (no timeseries)         : {rejected_counts['no_timeseries']}")

Loci passing all LAISS pre-filters : 404
  Rejected (catalog)               : 0
  Rejected (LC quality)            : 96
  Rejected (no timeseries)         : 0


In [ ]:
# ── Build a summary DataFrame from filtered loci ─────────────────────────
# Captures the ANTARES locus_id, coordinates, and key locus properties
# for downstream use (e.g. cross-matching with your reference CSV or
# feeding directly into the IsoForest pipeline).

rows = []
for locus in passed_loci:
    props = locus.properties
    rows.append({
        "locus_id"         : locus.locus_id,
        "ra"               : locus.ra,
        "dec"              : locus.dec,
        "num_mag_values"   : props.get("num_mag_values"),
        "newest_alert_mjd" : props.get("newest_alert_observation_time"),
        "oldest_alert_mjd" : props.get("oldest_alert_observation_time"),
        # ZTF object ID (if present) lets you join with Alerce / your CSVs
        "ztf_object_id"    : props.get("ztf_object_id"),
    })

df_antares = pd.DataFrame(rows)
print(f"df_antares shape: {df_antares.shape}")
df_antares.head()

### Load data

In [4]:
csv_path = "/Users/jennakempster-taylor/re-laiss/data/reference_20k_with_durations.csv" 
csv2_path = "/Users/jennakempster-taylor/re-laiss/data/reference_20k.csv"

# Load
df = pd.read_csv(csv_path, low_memory=False)
df2 = pd.read_csv(csv2_path, low_memory=False)

print("Shape of duration df:", df.shape)
print("Shape of previous df:", df2.shape)

# Grab our RELAISS features
default_lc_features = constants.lc_features_const.copy()
default_host_features = constants.host_features_const.copy()

# quick look at what is included
print("Default LC features (sample):", default_lc_features)
print("Default host features (sample):", default_host_features)

# look at additional headers of df comapred to df2:
additional = set(df.columns) - set(df2.columns)
print("Added columns in df:", additional)

Shape of duration df: (25515, 462)
Shape of previous df: (25515, 458)
Default LC features (sample): ['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude', 'g_skewness', 'r_skewness', 'g_beyond_2sigma', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate', 'g_max_rolling_variance', 'r_max_rolling_variance', 'g_mean_rolling_variance', 'r_mean_rolling_variance', 'g_rise_local_curvature', 'g_decline_local_curvature', 'r_rise_local_curvature', 'r_decline_local_curvature']
Default host features (sample): ['gKronMagCorrected', 'gKronRad', 'gExtNSigma', 'rKronMagCorrected', 'rKronRad', 'rExtNSigma', 'iKronMagCorrected', 'iKronRad', 'iExtNSigma', 'zKronMagCorrected', 'zKronRad', 'zExtNSigma', 'gminusrKronMag', 'rminusiKronMag', 'iminuszKronMag', 'rmomentXX', 'rmomentXY', 'rmomentYY']
Added columns in df: {'duration_days', 'antares_oldest_alert', 'antar

In [7]:
# Examine how much data in antares and duration:
total = len(df)
missing = df['antares_duration'].isna().sum()
present = total - missing
print("Amount missing:", missing)
print("Amount present:", present)
print( missing / present * 100,"%" )

Amount missing: 3756
Amount present: 21759
17.26182269405763 %


Much better than previous data 

In [9]:
added_cols = ['antares_duration', 'duration_days', 'antares_newest_alert', 'antares_oldest_alert']

missing_summary = (
    df[added_cols].isna().sum().to_frame('Missing')
    .assign(Total=len(df))
    .assign(Percent=lambda x: x['Missing'] / x['Total'] * 100)
)

print(missing_summary)


                      Missing  Total    Percent
antares_duration         3756  25515  14.720752
duration_days            3756  25515  14.720752
antares_newest_alert       48  25515   0.188125
antares_oldest_alert       48  25515   0.188125


### Cutting here

In [11]:
# Now build mask to apply to data to remove things that are not supernovae

mask = df['duration_days'] <= 200

df_200cut = df[mask].copy()
print(f"Cut dataset with objects of over 200 days alert span: {len(df_200cut)} rows (out of {len(df)})")

Cut dataset with objects of over 200 days alert span: 19027 rows (out of 25515)


In [17]:
 USE_HOST = True  # use only light curves initially

def overlap(cols, frame):
    # keep only columns present and drop *_err 
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df_200cut)
host_cols = overlap(default_host_features, df_200cut) if USE_HOST else []
feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in filtered data.")
print(f"Total candidate features: {len(feature_cols)}")

print(lc_cols)
print(host_cols)

Found 25 LC features in filtered data.
Found 11 host features in filtered data.
Total candidate features: 36
['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude', 'g_skewness', 'r_skewness', 'g_beyond_2sigma', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate', 'g_max_rolling_variance', 'r_max_rolling_variance', 'g_mean_rolling_variance', 'r_mean_rolling_variance', 'g_rise_local_curvature', 'g_decline_local_curvature', 'r_rise_local_curvature', 'r_decline_local_curvature']
['gKronRad', 'gExtNSigma', 'rKronRad', 'rExtNSigma', 'iKronRad', 'iExtNSigma', 'zKronRad', 'zExtNSigma', 'rmomentXX', 'rmomentXY', 'rmomentYY']


In [24]:
from sklearn.impute import KNNImputer

# Ensure all is nnumeric:
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_200cut[c])]
X = df_200cut[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print(f"X shape: {X.shape}  -> after KNN impute: {X_imp.shape}")

X shape: (19027, 36)  -> after KNN impute: (19027, 36)


In [25]:
iso = IsolationForest(
    n_estimators=300,
    contamination="auto",   # use auto for first run
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

Estimated anomaly rate: 0.039627897198717614


In [26]:
# Attach to df_filt (same length as scores/anomaly/rank)
df_filt = df_200cut.copy()
df_filt["iso_score"] = np.asarray(scores).ravel()
df_filt["iso_anomaly"] = np.asarray(anomaly).ravel()
df_filt["iso_rank"] = np.asarray(rank).ravel()

# Build 'out' from df_filt to keep lengths consistent
out = df_filt[["ZTFID", "r_duration_above_half_flux", "iso_score", "iso_anomaly", "iso_rank"]].copy()

# Optional context columns (from df_filt!)
context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
                if c in df_filt.columns]

preview = pd.concat(
    [
        df_filt[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True)
    ],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))

,ZTFID,t0,r_duration_above_half_flux,g_peak_mag,r_peak_mag,mean_g-r,features_valid,r_duration_above_half_flux,iso_score,iso_anomaly,iso_rank
7292,ZTF21acpfndw,59531.487442,0.101817,15.838621,15.937201,0.057640,False,0.101817,-0.180075,1,1
487,ZTF20aapchqy,58898.530648,121.733056,17.626400,17.099001,0.831728,False,121.733056,-0.173024,1,2
9204,ZTF22aauurbv,59782.285174,32.911817,15.255600,15.126600,0.978501,True,32.911817,-0.163391,1,3
1746,ZTF20actzmzp,59180.457222,0.997963,12.346022,12.654753,0.023451,False,0.997963,-0.158348,1,4
9847,ZTF22abewydn,59822.504873,18.979757,14.723200,15.154900,0.778043,False,18.979757,-0.157996,1,5
10730,ZTF22abqajav,59877.116910,19.000174,15.858900,15.849000,-0.186077,True,19.000174,-0.153038,1,6
15225,ZTF24aahkzvn,60389.519734,1.007234,16.399099,15.646200,0.219078,False,1.007234,-0.143582,1,7
12134,ZTF23aahfxpr,60057.302303,22.984803,16.141100,16.027000,0.509367,True,22.984803,-0.143296,1,8
5614,ZTF21abotogu,59423.217025,1.965208,16.244038,16.649410,-1.036775,False,1.965208,-0.133997,1,9
2773,ZTF21aagtqna,59248.514954,110.840567,18.550900,18.390396,0.690066,True,110.840567,-0.133810,1,10


### Examine anomalies PCA

Looks a bit off on the sclae so look at normalising beforehand:

In [50]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Standardise (zero mean, unit variance)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_filt)

# PCA on the scaled data
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

#Plot with colors for anomalies
colors = (df_filt["iso_anomaly"] == -1).astype(int)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1],
            c=colors, cmap="coolwarm", alpha=0.6)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Isolation Forest anomalies in PCA feature space (standardized)")
plt.show()
print("Explained variance ratio:", pca.explained_variance_ratio_)



ValueError: could not convert string to float: 'ZTF17aabtvsy'

In [ ]:
pca_components = pd.DataFrame(
    pca.components_,
    columns=X_imp_df.columns,
    index=['PCA1', 'PCA2']
)
pca_components.T.sort_values('PCA1', ascending=False).head(10)


In [ ]:
# Check variance levels 
pca_full = PCA().fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

for i, v in enumerate(cum_var[:10], 1):
    print(f"PCA{i}: {v:.3f}")

In [ ]:
# If needed: pip install plotly
%pip install plotly

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px

# 1) Standardize features for PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp_df)

# 2) 3D PCA
pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X_scaled)
evr = pca3.explained_variance_ratio_
title = f"3D PCA (Explained variance: PC1={evr[0]:.2f}, PC2={evr[1]:.2f}, PC3={evr[2]:.2f})"

# 3) Build plotting frame (keep indices aligned!)
plot_df = df_filt.copy()
plot_df["PCA1"] = X_pca3[:, 0]
plot_df["PCA2"] = X_pca3[:, 1]
plot_df["PCA3"] = X_pca3[:, 2]

# groups: anomalies, least anomalous, and the rest
top10_idx = plot_df.sort_values("iso_rank").head(10).index
bottom10_idx = plot_df.sort_values("iso_rank").tail(10).index

group = np.full(len(plot_df), "All")
group[plot_df.index.isin(top10_idx)] = "Top-10 anomalies"
group[plot_df.index.isin(bottom10_idx)] = "Least anomalous 10"
plot_df["group"] = group

# 4) Interactive 3D scatter with hover
fig = px.scatter_3d(
    plot_df,
    x="PCA1", y="PCA2", z="PCA3",
    color="group",
    hover_name="ZTFID",
    hover_data=["iso_rank", "iso_anomaly"],  # add more cols if useful
    title=title,
)
fig.update_traces(marker=dict(size=4))
fig.show()


### Use Alerce API

In [ ]:
client = Alerce()

In [ ]:
top10_df = preview.sort_values("iso_rank").head(10).reset_index(drop=True)


def fetch_quick_summary(oid):
    obj   = client.query_object(oid, format="pandas")          # stats (ra, dec, ndet, first/last mjd, etc.)
    probs = client.query_probabilities(oid, format="pandas")   # lc & stamp classifier probabilities
    mags  = client.query_magstats(oid, format="pandas")        # per-band stats
    return obj, probs, mags

# example
oids = list(top10_df["ZTFID"])  # whatever holds your top 10
summaries = {oid: fetch_quick_summary(oid) for oid in oids}

In [ ]:
# Extract list of object IDs
oids = top10_df["ZTFID"].tolist()
print(oids)

from alerce.core import Alerce
client = Alerce()

for oid in oids:
    print(f"\nFetching data for {oid}...")
    
    # Full light curve (detections + non-detections)
    lc   = client.query_lightcurve(oid, format="pandas")
    dets = client.query_detections(oid, format="pandas")
    nond = client.query_non_detections(oid, format="pandas")
    
    # quick check
    print(f"{oid}: {len(dets)} detections, {len(nond)} non-detections")



### Refinements: cut out galactic plane

In [58]:
# Look at headers to see what info we have on position
# we have ra and dec

from astropy.coordinates import SkyCoord
import astropy.units as u

transient = SkyCoord(df["ra"], df["dec"], unit="deg")


In [61]:
# Apply cut to original df (e.g. start again)
df = pd.read_csv(csv_path, low_memory=False)

coords = SkyCoord(ra=df["ra"].values * u.deg,
                  dec=df["dec"].values * u.deg,
                  frame="icrs")

b = coords.galactic.b.deg  # Galactic latitude

# Duration cut
# mask_duration = df["duration_days"].isna() | (df["duration_days"] <= 200)
mask_duration = df["duration_days"] <= 200

# Galactic latitude cut exclude plane |b| < 15 degrees
LAT_CUT = 15.0
mask_lat = np.abs(b) >= LAT_CUT

# Combine both
mask = mask_duration & mask_lat


df_cut = df[mask].copy()

print(f"Filtered dataset: {len(df_cut)} rows (out of {len(df)})")
print(f"Removed due to long duration or near galactic plane: {(~mask).sum()}")


Filtered dataset: 17254 rows (out of 25515)
Removed due to long duration or near galactic plane: 8261


In [65]:
USE_HOST = False  # use only light curves initially

def overlap(cols, frame):
    # keep only columns present and drop *_err 
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df_cut)
host_cols = overlap(default_host_features, df_cut) if USE_HOST else []
feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in filtered data.")
print(f"Total candidate features: {len(feature_cols)}")


Found 25 LC features in filtered data.
Total candidate features: 25


In [67]:
from sklearn.impute import KNNImputer

# Ensure all is nnumeric:
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_cut[c])]
X = df_cut[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print(f"X shape: {X.shape}  -> after KNN impute: {X_imp.shape}")

X shape: (17254, 25)  -> after KNN impute: (17254, 25)


In [68]:
iso = IsolationForest(
    n_estimators=300,
    contamination="auto",   # use auto
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

Estimated anomaly rate: 0.04225107221513852


In [69]:
# Attach to df_filt (same length as scores/anomaly/rank)
df_fil = df_cut.copy()
df_fil["iso_score"] = np.asarray(scores).ravel()
df_fil["iso_anomaly"] = np.asarray(anomaly).ravel()
df_fil["iso_rank"] = np.asarray(rank).ravel()

# Build 'out' from df_filt to keep lengths consistent
out = df_fil[["ZTFID", "r_duration_above_half_flux", "iso_score", "iso_anomaly", "iso_rank"]].copy()

# Optional context columns (from df_filt!)
context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
                if c in df_fil.columns]

preview = pd.concat(
    [
        df_fil[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True)
    ],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))

,ZTFID,t0,r_duration_above_half_flux,g_peak_mag,r_peak_mag,mean_g-r,features_valid,r_duration_above_half_flux,iso_score,iso_anomaly,iso_rank
9031,ZTF22abewydn,59822.504873,18.979757,14.723200,15.154900,0.778043,False,18.979757,-0.214968,1,1
1620,ZTF20actzmzp,59180.457222,0.997963,12.346022,12.654753,0.023451,False,0.997963,-0.199994,1,2
13814,ZTF24aahkzvn,60389.519734,1.007234,16.399099,15.646200,0.219078,False,1.007234,-0.195116,1,3
2942,ZTF21aanehlz,59269.465845,6.994768,13.308370,13.583603,0.529780,False,6.994768,-0.191229,1,4
11777,ZTF23aapsuva,60119.405752,118.730058,18.977400,18.754801,0.223114,True,118.730058,-0.184165,1,5
466,ZTF20aapchqy,58898.530648,121.733056,17.626400,17.099001,0.831728,False,121.733056,-0.180580,1,6
8174,ZTF22aapkbkl,59751.427049,135.732234,18.993099,18.510201,0.476248,True,135.732234,-0.172130,1,7
12078,ZTF23aatmqvz,60141.426181,441.989340,20.155300,19.810200,0.543308,True,441.989340,-0.169166,1,8
7516,ZTF22aajjqti,59707.430162,5.976447,14.233615,14.221217,0.063208,False,5.976447,-0.168979,1,9
6673,ZTF21acpfndw,59531.487442,0.101817,15.838621,15.937201,0.057640,False,0.101817,-0.168397,1,10


In [70]:
from joblib import dump

ARTIFACTS_PATH = "iforest_artifacts.joblib"
dump(
    {
        "iso": iso, 
        "knn_imp": knn_imp, 
        "numeric_feature_cols": numeric_feature_cols
    },
    ARTIFACTS_PATH
)
print(f"Saved: {ARTIFACTS_PATH}")


Saved: iforest_artifacts.joblib


### Host features and galactic cut 

In [ ]:
USE_HOST = True  # use only light curves initially

def overlap(cols, frame):
    # keep only columns present and drop *_err 
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df_cut)
host_cols = overlap(default_host_features, df_cut) if USE_HOST else []
feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in filtered data.")
print(f"Total candidate features: {len(feature_cols)}")


# Ensure all is nnumeric:
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_cut[c])]
X = df_cut[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print(f"X shape: {X.shape}  -> after KNN impute: {X_imp.shape}")

iso = IsolationForest(
    n_estimators=300,
    contamination="auto",   # use auto for first run
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

# Attach to df_filt (same length as scores/anomaly/rank)
df_fil = df_cut.copy()
df_fil["iso_score"] = np.asarray(scores).ravel()
df_fil["iso_anomaly"] = np.asarray(anomaly).ravel()
df_fil["iso_rank"] = np.asarray(rank).ravel()

# Build 'out' from df_filt to keep lengths consistent
out = df_fil[["ZTFID", "r_duration_above_half_flux", "iso_score", "iso_anomaly", "iso_rank"]].copy()

# Optional context columns (from df_filt!)
context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
                if c in df_fil.columns]

preview = pd.concat(
    [
        df_fil[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True)
    ],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))

### Use Alerce API and pull 100 SN-like objects, run iso forest on these objs

### Look at PCA 'anomalies' and compare

### Test on ~200 transients from Alerce (without Host first)

In [ ]:
from joblib import load
from alerce.core import Alerce
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
# load artifacts from training ( USE_HOST=False) 
art = load("iforest_artifacts.joblib")
iso = art["iso"]
knn_imp = art["knn_imp"]
numeric_feature_cols = art["numeric_feature_cols"]  # must match training order

In [ ]:
from alerce.core import Alerce
alerce = Alerce()
# list classifiers
classifiers = alerce.query_classifiers(format="pandas")  # or json
print(classifiers)

# list classes for one classifier
classes = alerce.query_classes("lc_classifier", format="pandas")
print(classes)

In [ ]:
# fetch 1000 most recent "Transient" objects 
alerce = Alerce()
objs = alerce.query_objects(
    classifier="lc_classifier_top",
    class_name="Transient",
    probability=0.7,
    page_size=1000,
    order_by="lastmjd",
    order_mode="DESC",
    format="pandas"
).drop_duplicates("oid")

#objs = objs.drop_duplicates(subset="oid").reset_index(drop=True)
oids = objs["oid"].tolist()

In [ ]:
probs_stamp = alerce.query_probabilities(
    oids,                 # positional: list of OIDs
    "stamp_classifier",   # positional: classifier name
    "pandas"              # positional: output format
)

# 3) Pivot to wide and threshold on SN probability
wide = probs_stamp.pivot_table(
    index="oid", columns="class_name", values="probability", aggfunc="max"
).fillna(0.0)

# ensure SN column exists
if "SN" not in wide.columns:
    wide["SN"] = 0.0

wide["P_SN"] = wide["SN"]
cand = wide[wide["P_SN"] >= 0.7].index

# 4) keep only those OIDs (SNe by stamp classifier)
sn_objs = objs_top[objs_top["oid"].isin(cand)].copy()
print(f"Total Transients: {len(objs_top)}  |  Likely SNe (stamp P>=0.7): {len(sn_objs)}")
sn_objs.head()


print(f"Total objects returned: {len(objs_top)}")
print(f"Likely SNe (P_SN ≥ 0.7): {len(sn_objs)}")
sn_objs.head()

In [ ]:
# pull feature tables for those OIDs 
def fetch_features(oids, keep_placeholders=True):
    rows, bad = [], []
    for oid in tqdm(oids, desc="Fetching ALeRCE features"):
        try:
            f = alerce.query_features(oid, format="pandas")
            if (f is None) or f.empty or not {"oid","name","value"}.issubset(f.columns):
                if keep_placeholders:
                    # one placeholder row; we'll add missing cols later
                    rows.append(pd.DataFrame(index=pd.Index([oid], name="oid")))
                else:
                    bad.append((oid, "empty-or-missing-cols"))
                continue
            fw = f.pivot_table(index="oid", columns="name", values="value", aggfunc="first")
            rows.append(fw)
        except Exception as e:
            if keep_placeholders:
                rows.append(pd.DataFrame(index=pd.Index([oid], name="oid")))
            else:
                bad.append((oid, str(e)))
    out = pd.concat(rows, axis=0) if rows else pd.DataFrame()
    return out, bad

df_new, bad = fetch_features(oids, keep_placeholders=True)

# Ensure every training column exists, then order them
missing = [c for c in numeric_feature_cols if c not in df_new.columns]
for c in missing:
    df_new[c] = np.nan
X_new = df_new[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)

# guardrails: drop rows that are *entirely* NaN if you prefer
# X_new = X_new.loc[~X_new.isna().all(axis=1)]

print(f"ALeRCE rows fetched: {len(df_new)}  usable after drop: {len(X_new)}  bad: {len(bad)}")


In [ ]:
# apply same imputer and model from training
X_new_imp = knn_imp.transform(X_new)
scores_new = iso.decision_function(X_new_imp)  # higher = more "normal"
pred_new   = iso.predict(X_new_imp)            # -1 outlier, +1 inlier
anom_new   = (pred_new == -1).astype(int)

In [ ]:
# package results
res = pd.DataFrame({
    "oid": df_new.index,
    "iso_score": scores_new,
    "iso_anomaly": anom_new
}).reset_index(drop=True)

In [ ]:
# attach some handy object metadata
keep_cols = [c for c in ["oid","meanra","meandec","ndet","firstmjd","lastmjd","class_name"] if c in objs.columns]
res = res.merge(objs[keep_cols], on="oid", how="left")


### rank by anomaly (lowest score = most anomalous)
res["iso_rank"] = pd.Series(res["iso_score"]).rank(method="first", ascending=True).astype(int)

res.sort_values("iso_rank").to_csv("alerce_200_recent_iforest.csv", index=False)
print("Saved: alerce_200_recent_iforest.csv")
res.sort_values("iso_rank").head(10)
